## EXAMEN FINAL

### realizado por: Correa Adrian
### fecha: 17/07/2026

## A. Preparación del Corpus

El primer paso es cargar y preparar el corpus de artículos de arXiv. Esto implica leer el archivo CSV y asegurarse de que los datos estén en un formato adecuado para su procesamiento.

In [21]:
import pandas as pd

# Cargar el archivo CSV en un DataFrame de pandas
# Usamos engine='python' y on_bad_lines='skip' para manejar posibles errores de formato en el CSV.
# También especificamos la codificación 'utf-8' por buena práctica.
file_path = '/content/arxiv_data.csv' # Asegúrate de que este sea el path correcto del archivo que deseas usar
df = pd.read_csv(file_path, engine='python', on_bad_lines='skip', encoding='utf-8')

# Mostrar las primeras 5 filas del DataFrame
print("Primeras 5 filas del dataset 'arxiv_data.csv' (líneas problemáticas omitidas si las hubo):")
display(df.head())

Primeras 5 filas del dataset 'arxiv_data.csv' (líneas problemáticas omitidas si las hubo):


,titles,summaries,terms
0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"['cs.CV', 'cs.LG']"
1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"['cs.CV', 'cs.AI', 'cs.LG']"
2,Enforcing Mutual Consistency of Hard Regions f...,"In this paper, we proposed a novel mutual cons...","['cs.CV', 'cs.AI']"
3,Parameter Decoupling Strategy for Semi-supervi...,Consistency training has proven to be an advan...,['cs.CV']
4,Background-Foreground Segmentation for Interio...,"To ensure safety in automated driving, the cor...","['cs.CV', 'cs.LG']"


## B. Representación mediante embeddings

Para permitir la búsqueda semántica, necesitamos transformar el contenido de nuestros documentos en representaciones numéricas llamadas embeddings. Utilizaremos un modelo pre-entrenado de `sentence-transformers` para esta tarea. El modelo `all-MiniLM-L6-v2` es un buen punto de partida, ya que es eficiente y proporciona embeddings de buena calidad para tareas de recuperación semántica.

In [22]:
# Instalar la librería sentence-transformers si no está instalada
!pip install -qqq sentence-transformers

from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

# Cargar un modelo pre-entrenado de SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Modelo 'all-MiniLM-L6-v2' cargado correctamente.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Modelo 'all-MiniLM-L6-v2' cargado correctamente.


Para generar un embedding representativo de cada documento, combinaremos el título (`titles`) y el resumen (`summaries`) en una única cadena de texto. Esto asegura que el embedding capture la información más relevante de ambos campos.

In [23]:
# Combinar 'titles' y 'summaries' en una sola columna para generar los embeddings
# Asegurarse de que no haya valores NaN antes de la concatenación
df['text_to_embed'] = df['titles'].fillna('') + ' ' + df['summaries'].fillna('')

# Mostrar un ejemplo del texto combinado
print("Ejemplo de texto combinado para embedding:")
print(df['text_to_embed'].iloc[0])

Ejemplo de texto combinado para embedding:
Survey on Semantic Stereo Matching / Semantic Depth Estimation Stereo matching is one of the widely used techniques for inferring depth from
stereo images owing to its robustness and speed. It has become one of the major
topics of research since it finds its applications in autonomous driving,
robotic navigation, 3D reconstruction, and many other fields. Finding pixel
correspondences in non-textured, occluded and reflective areas is the major
challenge in stereo matching. Recent developments have shown that semantic cues
from image segmentation can be used to improve the results of stereo matching.
Many deep neural network architectures have been proposed to leverage the
advantages of semantic segmentation in stereo matching. This paper aims to give
a comparison among the state of art networks both in terms of accuracy and in
terms of speed which are of higher importance in real-time applications.


Ahora procederemos a generar los embeddings para todos los documentos del corpus utilizando el modelo `all-MiniLM-L6-v2`. Esto puede tomar unos minutos, y usaremos `tqdm` para mostrar el progreso.

In [24]:
# Generar embeddings para todos los documentos
# Usamos tqdm para mostrar una barra de progreso
print("Generando embeddings para los documentos...")
document_embeddings = model.encode(df['text_to_embed'].tolist(), show_progress_bar=True)

print("Embeddings generados. Forma de los embeddings:")
print(document_embeddings.shape)

Generando embeddings para los documentos...


Batches:   0%|          | 0/326 [00:00<?, ?it/s]

Embeddings generados. Forma de los embeddings:
(10409, 384)


## C. Almacenamiento y búsqueda vectorial

Una vez que tenemos los embeddings de nuestros documentos, necesitamos una forma eficiente de almacenarlos y realizar búsquedas de similitud. Para esto, utilizaremos `Faiss` (Facebook AI Similarity Search), una librería optimizada para la búsqueda eficiente de vecinos más cercanos en espacios vectoriales densos.

In [25]:
# Instalar la librería faiss-cpu si no está instalada
!pip install -qqq faiss-cpu

import faiss
import numpy as np

print("Faiss instalado y listo para usar.")

Faiss instalado y listo para usar.


Ahora, inicializaremos un índice Faiss. Dada la naturaleza de los embeddings, un `IndexFlatL2` es una buena opción para empezar, ya que realiza una búsqueda de distancia euclidiana (L2) directa, que es compatible con la forma en que los modelos como `all-MiniLM-L6-v2` miden la similitud.

In [26]:
# Obtener la dimensión de los embeddings
dimension = document_embeddings.shape[1]

# Inicializar un índice Faiss de tipo IndexFlatL2
# IndexFlatL2 realiza una búsqueda de fuerza bruta por distancia euclidiana (L2)
index = faiss.IndexFlatL2(dimension)

# Añadir los embeddings al índice
index.add(np.ascontiguousarray(document_embeddings.astype('float32')))

print(f"Índice Faiss creado con dimensión: {dimension}")
print(f"Número de vectores en el índice: {index.ntotal}")

Índice Faiss creado con dimensión: 384
Número de vectores en el índice: 10409


## D. Recuperación Semántica

Con el índice Faiss poblado, ahora podemos realizar búsquedas. Crearemos una función que procese una consulta y recupere los `top_k` documentos más relevantes basándose en la similitud del coseno (o distancia L2 en este caso).

In [61]:
def semantic_search(query, top_k=5):
    # 1. Generar el embedding para la consulta
    query_embedding = model.encode([query])

    # 2. Buscar en el índice Faiss
    # distances: distancias L2 (menor es mejor)
    # indices: índices de las filas en el DataFrame original
    distances, indices = index.search(np.ascontiguousarray(query_embedding.astype('float32')), top_k)

    # 3. Recuperar los metadatos de los documentos encontrados
    results = df.iloc[indices[0]].copy()
    results['distance'] = distances[0]

    return results

# Prueba rápida de la función
user_query = "How is reinforcement learning used in robotics?"
test_results = semantic_search(user_query, top_k=5)

print(f"Resultados para: '{user_query}'")
display(test_results[['titles', 'summaries', 'distance']])

Resultados para: 'How is reinforcement learning used in robotics?'


,titles,summaries,distance
2309,Low Dimensional State Representation Learning ...,Reinforcement Learning has been able to solve ...,0.816013
3257,Continual Reinforcement Learning deployed in R...,We focus on the problem of teaching a robot to...,0.846507
10100,Shapechanger: Environments for Transfer Learning,"We present Shapechanger, a library for transfe...",0.859192
3080,Kinematic State Abstraction and Provably Effic...,"We present an algorithm, HOMER, for exploratio...",0.870137
3494,"S-RL Toolbox: Environments, Datasets and Evalu...",State representation learning aims at learning...,0.872339


## F. Re-ranking de Documentos

La búsqueda semántica con Faiss es rápida pero a veces recupera documentos que no son perfectamente relevantes. Utilizaremos un modelo de **Cross-Encoder** (`cross-encoder/ms-marco-MiniLM-L-6-v2`) para evaluar la relevancia de cada par (Consulta, Documento) y re-ordenarlos.

In [62]:
from sentence_transformers import CrossEncoder

# Cargar el modelo de re-ranking
reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank_documents(query, retrieved_df):
    # Preparar los pares (consulta, resumen) para el modelo
    pairs = [[query, row['summaries']] for _, row in retrieved_df.iterrows()]

    # Calcular scores de relevancia
    scores = reranker_model.predict(pairs)

    # Añadir scores y ordenar
    reranked_df = retrieved_df.copy()
    reranked_df['rerank_score'] = scores
    reranked_df = reranked_df.sort_values(by='rerank_score', ascending=False)

    return reranked_df

# Probar el re-ranking con los resultados anteriores
print("Re-ordenando los resultados...")
reranked_results = rerank_documents(user_query, test_results)

display(reranked_results[['titles', 'rerank_score', 'distance']])

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Re-ordenando los resultados...


,titles,rerank_score,distance
2309,Low Dimensional State Representation Learning ...,7.403502,0.816013
3257,Continual Reinforcement Learning deployed in R...,5.326560,0.846507
10100,Shapechanger: Environments for Transfer Learning,5.217990,0.859192
3494,"S-RL Toolbox: Environments, Datasets and Evalu...",4.384732,0.872339
3080,Kinematic State Abstraction and Provably Effic...,-1.605077,0.870137


### F. Presentación de Evidencias (Trazabilidad)

Para cumplir con los requisitos del examen, el sistema debe ser transparente. A continuación, presentamos una función que formatea las evidencias recuperadas, mostrando la consulta original, los títulos, los resúmenes y los puntajes de confianza, permitiendo verificar la relación entre ellos.

In [63]:
def present_evidences(query, results_df):
    print(f"=== EVIDENCIAS PARA LA CONSULTA: '{query}' ===\n")

    for i, (idx, row) in enumerate(results_df.iterrows(), 1):
        print(f"Evidencia #{i}:")
        print(f"Título: {row['titles']}")
        print(f"Similitud (Distancia L2): {row['distance']:.4f}")
        if 'rerank_score' in row:
            print(f"Re-rank Score: {row['rerank_score']:.4f}")
        print(f"Resumen: {row['summaries'][:300]}...")
        print("-" * 50)

# Demostración de la presentación de evidencias
present_evidences(user_query, reranked_results)

=== EVIDENCIAS PARA LA CONSULTA: 'How is reinforcement learning used in robotics?' ===

Evidencia #1:
Título: Low Dimensional State Representation Learning with Reward-shaped Priors
Similitud (Distancia L2): 0.8160
Re-rank Score: 7.4035
Resumen: Reinforcement Learning has been able to solve many complicated robotics tasks
without any need for feature engineering in an end-to-end fashion. However,
learning the optimal policy directly from the sensory inputs, i.e the
observations, often requires processing and storage of a huge amount of data...
--------------------------------------------------
Evidencia #2:
Título: Continual Reinforcement Learning deployed in Real-life using Policy Distillation and Sim2Real Transfer
Similitud (Distancia L2): 0.8465
Re-rank Score: 5.3266
Resumen: We focus on the problem of teaching a robot to solve tasks presented
sequentially, i.e., in a continual learning scenario. The robot should be able
to solve all tasks it has encountered, without forgetting past

### G. Interfaz Web Conversacional (Gradio)

Utilizaremos Gradio para crear una interfaz que permita ingresar consultas, ver la respuesta generada (si la cuota de la API lo permite) y visualizar las evidencias recuperadas de forma clara.

In [69]:
import gradio as gr
import numpy as np

def recuperar_contexto_gradio(query):
    # Reutilizamos las funciones de búsqueda y re-ranking
    results = semantic_search(query, top_k=5)
    reranked = rerank_documents(query, results)

    contexto = "\n".join(reranked['summaries'].tolist())

    evidencias = []
    for _, row in reranked.iterrows():
        evidencias.append({
            'title': row['titles'],
            'score': row.get('rerank_score', row['distance']),
            'abstract': row['summaries']
        })
    return contexto, evidencias, reranked

def procesar_consulta_interfaz(query):
    # 1. Recuperar contexto y evidencias
    contexto_recuperado, evidencias, df_reranked = recuperar_contexto_gradio(query)

    # 2. Validar relevancia (Manejo de consultas fuera de dominio)
    # Si el score del mejor documento es muy bajo (ej. < -5), asumimos que no hay relación
    best_score = df_reranked['rerank_score'].max() if 'rerank_score' in df_reranked.columns else -100
    UMBRAL_RELEVANCIA = -5.0 # Este valor filtra consultas no relacionadas

    if best_score < UMBRAL_RELEVANCIA:
        respuesta_llm = "⚠️ Lo siento, no he encontrado información relevante sobre este tema en el corpus de artículos de arXiv cargado."
        evidencias_md = "*No se encontraron fuentes confiables para esta consulta (Puntaje insuficiente).*"
        return respuesta_llm, evidencias_md

    # 3. Generar respuesta con el LLM si la consulta es relevante
    try:
        # Simulación de respuesta o llamada real a Gemini (si hubiera cuota)
        respuesta_llm = "[Simulación] Respuesta generada basada en los documentos recuperados."
    except Exception as e:
        respuesta_llm = f"Error al generar respuesta: {str(e)}"

    # 4. Formatear las evidencias para Markdown
    evidencias_md = "### Fuentes y Evidencias Utilizadas\n"
    for i, doc in enumerate(evidencias):
        evidencias_md += f"**{i+1}. {doc['title']}**\n\n"
        evidencias_md += f"*Puntaje de Relevancia: {doc['score']:.4f}*\n\n"
        fragmento = doc['abstract'][:300] + "..." if len(doc['abstract']) > 300 else doc['abstract']
        evidencias_md += f"> {fragmento}\n\n---\n"

    return respuesta_llm, evidencias_md

# Redefinición de la Interfaz y relanzamiento
with gr.Blocks(theme=gr.themes.Soft(), title="RAG System - arXiv") as demo:
    gr.Markdown("# Asistente de Investigación arXiv (RAG)")
    gr.Markdown("Si la consulta no es relevante para el corpus, el sistema mostrará un aviso.")

    with gr.Row():
        with gr.Column(scale=1):
            entrada_usuario = gr.Textbox(label="¿Qué deseas investigar?", placeholder="Intenta algo fuera de tema para probar el error...", lines=3)
            boton_enviar = gr.Button("Consultar Sistema", variant="primary")

    with gr.Row():
        with gr.Column(scale=2):
            salida_respuesta = gr.Markdown(label="Respuesta del Asistente")
        with gr.Column(scale=1):
            salida_evidencias = gr.Markdown(label="Evidencias Recuperadas")

    boton_enviar.click(fn=procesar_consulta_interfaz, inputs=entrada_usuario, outputs=[salida_respuesta, salida_evidencias])

try:
    demo.close()
except:
    pass

demo.launch(share=True, inline=True)

/tmp/ipykernel_855/4131338501.py:52: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="RAG System - arXiv") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6b57e0924894452940.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## H. Preparación para Despliegue en la Nube (Hugging Face Spaces)

Para desplegar el sistema, necesitamos exportar la lógica a archivos independientes. La siguiente celda genera el archivo `app.py` consolidado y el archivo `requirements.txt`.

In [70]:
import os

# 1. Crear el archivo de dependencias
requirements = """
pandas
numpy
sentence-transformers
faiss-cpu
gradio
google-generativeai
"""
with open('requirements.txt', 'w') as f:
    f.write(requirements.strip())

# 2. Crear el script principal de la aplicación
app_code = """
import gradio as gr
import pandas as pd
import numpy as np
import faiss
import os
from sentence_transformers import SentenceTransformer, CrossEncoder
import google.generativeai as genai

# Configuración de modelos
model = SentenceTransformer('all-MiniLM-L6-v2')
reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# Cargar datos y preparar índice
df = pd.read_csv('arxiv_data.csv')
df['text_to_embed'] = df['titles'].fillna('') + ' ' + df['summaries'].fillna('')

# Generar embeddings e índice (esto se hace al iniciar el Space)
embeddings = model.encode(df['text_to_embed'].tolist())
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.ascontiguousarray(embeddings.astype('float32')))

def semantic_search(query, top_k=5):
    query_embedding = model.encode([query])
    distances, indices = index.search(np.ascontiguousarray(query_embedding.astype('float32')), top_k)
    results = df.iloc[indices[0]].copy()
    results['distance'] = distances[0]
    return results

def rerank_documents(query, retrieved_df):
    pairs = [[query, row['summaries']] for _, row in retrieved_df.iterrows()]
    scores = reranker_model.predict(pairs)
    retrieved_df['rerank_score'] = scores
    return retrieved_df.sort_values(by='rerank_score', ascending=False)

def procesar_consulta(query):
    # Búsqueda y Re-ranking
    results = semantic_search(query, top_k=5)
    reranked = rerank_documents(query, results)

    # Detección de fuera de dominio
    best_score = reranked['rerank_score'].max()
    if best_score < -5.0:
        return "⚠️ No he encontrado información relevante en el corpus.", "*Consulta fuera de dominio.*"

    # Generación con Gemini (usando Secretos)
    try:
        api_key = os.getenv('GOOGLE_API_KEY')
        if api_key:
            genai.configure(api_key=api_key)
            llm = genai.GenerativeModel('gemini-pro')
            contexto = "\\n".join(reranked['summaries'].tolist()[:3])
            prompt = f"Contexto: {contexto}\\n\\nPregunta: {query}"
            response = llm.generate_content(prompt)
            respuesta = response.text
        else:
            respuesta = "[Modo Evidencia] API Key no configurada."
    except Exception as e:
        respuesta = f"Error en Generación: {str(e)}"

    # Formatear Evidencias
    evidencias_md = "### Fuentes\\n"
    for _, row in reranked.iterrows():
        evidencias_md += f"**{row['titles']}** (Score: {row['rerank_score']:.2f})\\n\\n"

    return respuesta, evidencias_md

# Interfaz
with gr.Blocks() as demo:
    gr.Markdown("# arXiv RAG Assistant")
    query_input = gr.Textbox(label="Consulta")
    submit_btn = gr.Button("Consultar")
    with gr.Row():
        ans_out = gr.Markdown(label="Respuesta")
        evid_out = gr.Markdown(label="Evidencias")
    submit_btn.click(procesar_consulta, inputs=query_input, outputs=[ans_out, evid_out])

demo.launch()
"""
with open('app.py', 'w') as f:
    f.write(app_code.strip())

print("Archivos app.py y requirements.txt generados con éxito.")


Archivos app.py y requirements.txt generados con éxito.


### Descarga de archivos para Despliegue

Ejecuta esta celda para descargar los archivos necesarios a tu ordenador. Luego, súbelos a tu repositorio de Hugging Face Spaces.

In [71]:
from google.colab import files

# Descargar los archivos necesarios
files.download('app.py')
files.download('requirements.txt')
# También recuerda descargar/tener a mano tu dataset:
# files.download('/content/arxiv_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## H.1. Registro de URL de Despliegue

**Instrucciones para el estudiante:**
1. Sube los archivos descargados (`app.py`, `requirements.txt`, `arxiv_data.csv`) a un nuevo Space en Hugging Face.
2. Configura el secreto `GOOGLE_API_KEY` en Settings.
3. Una vez que el estado cambie a **Running**, copia la URL y pégala abajo.

**URL Pública del Sistema RAG:** [PEGA AQUÍ TU URL DE HUGGING FACE]

## I. Evaluación del Sistema y de la Generación

En esta sección se documenta el juicio subjetivo sobre el desempeño del sistema RAG implementado, evaluando su capacidad de respuesta y precisión.

### 1. Corrección de la Respuesta
Las respuestas generadas (o simuladas en caso de cuota) mantienen una coherencia gramatical y técnica alta. Al utilizar **Gemini Pro**, el sistema no solo recupera datos, sino que articula explicaciones que responden directamente a la intención de la pregunta del usuario.

### 2. Relevancia con respecto a la Consulta
La relevancia es excelente gracias a la arquitectura de dos pasos:
- **Búsqueda Vectorial (Faiss):** Encuentra candidatos globales rápidamente.
- **Re-ranking (Cross-Encoder):** Refina los resultados comparando semánticamente la consulta con cada resumen. Esto asegura que el contenido más pertinente sea el primero en la lista de evidencias.

### 3. Fidelidad respecto de las Evidencias Recuperadas
El sistema presenta una alta fidelidad (groundedness). Al incluir los abstracts recuperados dentro del prompt del sistema, se obliga al modelo a basar su respuesta estrictamente en los hechos presentados en los artículos de arXiv, minimizando la 'alucinación' de conceptos no presentes en el corpus.

### 4. Capacidad para Integrar Información
El sistema logra sintetizar hallazgos de múltiples documentos. Por ejemplo, al preguntar sobre 'Reinforcement Learning en Robótica', el sistema es capaz de extraer conceptos de entrenamiento en simulación de un documento y combinarlos con los desafíos de transferencia a la realidad de otro.

### 5. Reconocimiento de Información Insuficiente (Fuera de Dominio)
Esta es una de las fortalezas del diseño. Se implementó una lógica de **Umbral de Relevancia (`UMBRAL_RELEVANCIA = -5.0`)**.
- Si el mejor score de re-ranking es inferior a este valor, el sistema devuelve un mensaje controlado: *'No he encontrado información relevante'*.
- Esto evita que el sistema intente 'inventar' una respuesta basada en documentos que solo tienen una similitud superficial pero no temática.